In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "llmware/slim-summary-tiny"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

text = """
The Apollo 11 mission was the first manned mission to land on the Moon. 
It launched on July 16, 1969. The crew consisted of Neil Armstrong, Buzz Aldrin, and Michael Collins. 
Armstrong became the first human to step onto the lunar surface on July 20.
"""

# The model expects a specific prompt format for function calls
prompt = "<human>: " + text + "\n<function> summarize points (3) </function>\n<bot>:"

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(
    **inputs, 
    max_new_tokens=200, 
    temperature=0.3,         # Lower temperature makes it more focused
    repetition_penalty=1.2,  # <--- This stops the "The mission was... The mission was..." loop
    do_sample=True
)

# Decode only the new tokens
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)



print(response)
# Output example: ['Launched July 16, 1969', 'Crew: Armstrong, Aldrin, Collins', 'Landed July 20']

["Apollo 11 Mission", "Launched on July 16, 1969"]


In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 'flan-t5-base' is the sweet spot for size vs. performance. 
# You can try 'flan-t5-small' (80M params) if this is still too big.
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text = """
In Q3, our team rolled out a new customer onboarding flow aimed at reducing time-to-value for small and mid-sized clients. We replaced the old “one-size-fits-all” setup with a guided checklist, added in-app tooltips for the first five key actions, and introduced a short welcome call for accounts over a certain contract size. Early feedback suggests new users feel less overwhelmed, but some still get stuck when connecting third-party integrations.

On the operations side, we tightened up how we handle incidents. We defined severity levels more clearly, set up a rotating on-call schedule, and added a “first 15 minutes” runbook to speed up triage. Since the change, we’ve seen faster initial response times, though a few incidents still take too long to fully resolve because ownership between product and platform isn’t always obvious.

We also made progress on data quality. We implemented basic validation checks at ingestion, started tracking a small set of “golden metrics,” and created a dashboard that highlights missing or unusual values. This reduced the number of broken reports, but it also revealed gaps in how different teams define the same fields, which is now causing inconsistencies in downstream analysis.

Finally, we learned some useful lessons about delivery. When requirements were vague, work expanded and deadlines slipped; when we agreed success criteria up front, projects moved smoothly. For the next quarter, we plan to standardise kickoff templates, limit work-in-progress to keep focus, and run two customer interviews per month to ensure we’re building the right things.
"""

# We give it a pattern to follow so it knows it needs multiple points
prompt = f"""
Summarize the text into exactly 3 bullet points.

Text: {text}

Summary:
- 
"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs, 
    max_length=300, 
    min_length=60,           # FORCE it to write at least 30 tokens
    repetition_penalty=1.5,  # Penalize repeating the same sentence
    num_beams=4              # Search harder for the best answer
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

New customer onboarding flow aimed at reducing time-to-value for small and mid-sized clients. We’ve made progress on data quality, but we’ve also learned some useful lessons about delivery. The next quarter, we plan to standardise kickoff templates, limit work-in-progress to keep focus, and run two customer interviews per month to ensure we’re building the right things.


In [12]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

# --- STAGE 1: The Summarizer (DistilBART) ---
# Goal: Get the core info out of the noise.
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

# --- STAGE 2: The Formatter (FLAN-T5-Small) ---
# Goal: Rewrite the summary into a clean list.
formatter_name = "google/flan-t5-base"
formatter_tokenizer = AutoTokenizer.from_pretrained(formatter_name)
formatter_model = AutoModelForSeq2SeqLM.from_pretrained(formatter_name)

# Input Text
text = """
In Q3, our team rolled out a new customer onboarding flow aimed at reducing time-to-value for small and mid-sized clients. We replaced the old “one-size-fits-all” setup with a guided checklist, added in-app tooltips for the first five key actions, and introduced a short welcome call for accounts over a certain contract size. Early feedback suggests new users feel less overwhelmed, but some still get stuck when connecting third-party integrations.

On the operations side, we tightened up how we handle incidents. We defined severity levels more clearly, set up a rotating on-call schedule, and added a “first 15 minutes” runbook to speed up triage. Since the change, we’ve seen faster initial response times, though a few incidents still take too long to fully resolve because ownership between product and platform isn’t always obvious.

We also made progress on data quality. We implemented basic validation checks at ingestion, started tracking a small set of “golden metrics,” and created a dashboard that highlights missing or unusual values. This reduced the number of broken reports, but it also revealed gaps in how different teams define the same fields, which is now causing inconsistencies in downstream analysis.

Finally, we learned some useful lessons about delivery. When requirements were vague, work expanded and deadlines slipped; when we agreed success criteria up front, projects moved smoothly. For the next quarter, we plan to standardise kickoff templates, limit work-in-progress to keep focus, and run two customer interviews per month to ensure we’re building the right things.

"""

# --- EXECUTION ---

# Step 1: Generate the dense paragraph summary
# We ask for a slightly longer summary so we have enough meat to make a list.
summary_output = summarizer(text, max_length=150, min_length=50, do_sample=False)
dense_summary = summary_output[0]['summary_text']

print(f"--- Intermediate Stage (Paragraph) ---\n{dense_summary}\n")

# Step 2: Ask FLAN-T5 to reformat it
# We explicitly tell it to convert the input into points.
format_prompt = f"Convert the following paragraph into a list of bullet points:\n\n{dense_summary}"

inputs = formatter_tokenizer(format_prompt, return_tensors="pt")

outputs = formatter_model.generate(
    **inputs, 
    max_length=200, 
    num_beams=5,             # High beams ensures it finds the structure
    early_stopping=True
)

final_list = formatter_tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"--- Final Stage (List) ---\n{final_list}")

Device set to use mps:0


--- Intermediate Stage (Paragraph) ---
 In Q3, our team rolled out a new customer onboarding flow aimed at reducing time-to-value for small and mid-sized clients . Early feedback suggests new users feel less overwhelmed, but some still get stuck when connecting third-party integrations .

--- Final Stage (List) ---
In Q3, our team rolled out a new customer onboarding flow aimed at reducing time-to-value for small and mid-sized clients . Early feedback suggests new users feel less overwhelmed but some still get stuck when connecting third-party integrations .


In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Load the specialized KeyBART model (~500MB)
model_name = "ml6team/keyphrase-generation-keybart-inspec"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text = """
In Q3, our team rolled out a new customer onboarding flow aimed at reducing time-to-value for small and mid-sized clients. We replaced the old “one-size-fits-all” setup with a guided checklist, added in-app tooltips for the first five key actions, and introduced a short welcome call for accounts over a certain contract size. Early feedback suggests new users feel less overwhelmed, but some still get stuck when connecting third-party integrations.

On the operations side, we tightened up how we handle incidents. We defined severity levels more clearly, set up a rotating on-call schedule, and added a “first 15 minutes” runbook to speed up triage. Since the change, we’ve seen faster initial response times, though a few incidents still take too long to fully resolve because ownership between product and platform isn’t always obvious.

We also made progress on data quality. We implemented basic validation checks at ingestion, started tracking a small set of “golden metrics,” and created a dashboard that highlights missing or unusual values. This reduced the number of broken reports, but it also revealed gaps in how different teams define the same fields, which is now causing inconsistencies in downstream analysis.

Finally, we learned some useful lessons about delivery. When requirements were vague, work expanded and deadlines slipped; when we agreed success criteria up front, projects moved smoothly. For the next quarter, we plan to standardise kickoff templates, limit work-in-progress to keep focus, and run two customer interviews per month to ensure we’re building the right things.

"""

# 2. Generate the keyphrases
inputs = tokenizer(text, return_tensors="pt")
outputs = model.generate(**inputs, max_length=100, do_sample=False)

# 3. Decode the raw output
# The model outputs raw text like: "Apollo 11; Neil Armstrong; Space Race"
raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 4. Convert to a Python List
# We split by the semicolon delimiter and strip whitespace
clean_list = [phrase.strip() for phrase in raw_output.split(';') if phrase.strip()]

# 5. Print results
print(f"Raw Output: {raw_output}\n")
print("--- Final List ---")
for point in clean_list:
    print(f"- {point}")

Raw Output: customer onboarding flow ; mid-sized clients ; guided checklist ; in-app tooltips ; contract size ; severity levels ; rotating on-call schedule ; first 15 minutes runbook ; tri

--- Final List ---
- customer onboarding flow
- mid-sized clients
- guided checklist
- in-app tooltips
- contract size
- severity levels
- rotating on-call schedule
- first 15 minutes runbook
- tri


In [16]:
# 2. Generate the keyphrases
inputs = tokenizer(text, return_tensors="pt")
outputs = model.generate(**inputs, max_length=100, do_sample=False)

# 3. Decode the raw output
# The model outputs raw text like: "Apollo 11; Neil Armstrong; Space Race"
raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 4. Convert to a Python List
# We split by the semicolon delimiter and strip whitespace
clean_list = [phrase.strip() for phrase in raw_output.split(';') if phrase.strip()]

# 5. Print results
print(f"Raw Output: {raw_output}\n")
print("--- Final List ---")
for point in clean_list:
    print(f"- {point}")

Raw Output: customer onboarding flow ; mid-sized clients ; guided checklist ; in-app tooltips ; contract size ; severity levels ; rotating on-call schedule ; first 15 minutes runbook ; tri

--- Final List ---
- customer onboarding flow
- mid-sized clients
- guided checklist
- in-app tooltips
- contract size
- severity levels
- rotating on-call schedule
- first 15 minutes runbook
- tri
